In [ ]:
import netrc
import requests
from tqdm import tqdm

# Change PRODUCT_NAME to the SAFE file you want to download.
PRODUCT_NAME = "S2A_MSIL1C_20250102T183751_N0511_R027_T11SLT_20250102T202910.SAFE"

# 1. Fetch Product ID
search_url = f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Name eq '{PRODUCT_NAME}'"
response = requests.get(search_url).json()

if not response.get("value"):
    raise ValueError(f"Product '{PRODUCT_NAME}' was not found.")

product_id = response["value"][0]["Id"]
download_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"

# 2. Authenticate
login, _, password = netrc.netrc().authenticators(
    "identity.dataspace.copernicus.eu"
)

token = requests.post(
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    data={
        "client_id": "cdse-public",
        "username": login,
        "password": password,
        "grant_type": "password",
    },
).json()["access_token"]

# 3. Stream with progress bar
with requests.get(
    download_url, headers={"Authorization": f"Bearer {token}"}, stream=True
) as r:
    r.raise_for_status()

    # Get total file size from response headers (in bytes)
    total_size = int(r.headers.get("content-length", 0))
    chunk_size = 8192  # 8 KB chunks

    with (
        open("output.zip", "wb") as f,
        tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Downloading",
        ) as bar,
    ):
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))